# Multi-Token Prediction Loss（多 Token 预测损失）

**难度：** Medium

**函数签名：** `multi_token_prediction_loss(hidden_states, heads, targets) -> Tensor`

**参数：**
- `hidden_states` — 隐藏状态 (B, S, D)
- `heads` — N 个权重矩阵列表，每个 (D, vocab_size)
- `targets` — 目标 token ID (B, S, N)

**返回：** 标量均值损失

**约束：** 手动实现交叉熵，不能用 F.cross_entropy

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def multi_token_prediction_loss(hidden_states, heads, targets):
    B, S, D = hidden_states.shape
    N = len(heads)
    total_loss = 0.0

    for i, head in enumerate(heads):
        # ---- Step 1: 计算 logits ----
        # hidden_states: (B, S, D)  head: (D, V)
        # logits: (B, S, V) — 每个位置对词表的得分
        logits = hidden_states @ head

        # ---- Step 2: 数值稳定的 log-softmax ----
        # 为什么先减最大值？
        # log(softmax(x)) = x - log(sum(exp(x)))
        # 如果 x 很大，exp(x) 会溢出 → NaN
        # 减去 max(x) 不改变结果（分子分母同除 exp(max)）但防止溢出
        logits_max = logits.max(dim=-1, keepdim=True).values
        shifted = logits - logits_max
        # log(softmax) = shifted - log(sum(exp(shifted)))
        log_probs = shifted - torch.log(torch.exp(shifted).sum(dim=-1, keepdim=True))

        # ---- Step 3: Gather 目标 token 的 log 概率 ----
        # targets[:, :, i] 形状: (B, S) — 第 i 个头的目标 token ID
        # .unsqueeze(-1) → (B, S, 1) — 用于 gather 的索引
        # gather(-1, idx) 在最后一维按 idx 取值
        # 结果 log_p 形状: (B, S)
        tgt = targets[:, :, i]
        log_p = log_probs.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)

        # ---- Step 4: 累加损失 ----
        # -log_p.mean() 是交叉熵损失
        # 所有位置的负对数似然取平均
        total_loss = total_loss + (-log_p.mean())

    # 所有头的损失取平均
    return total_loss / N

In [ ]:
torch.manual_seed(0)
B, S, D, V = 2, 5, 16, 10

hidden = torch.randn(B, S, D)
head_single = torch.randn(D, V)
targets_single = torch.randint(0, V, (B, S, 1))

# N=1 should match standard cross-entropy
mtp_loss = multi_token_prediction_loss(hidden, [head_single], targets_single)

logits = hidden @ head_single
ce_loss = torch.nn.functional.cross_entropy(logits.reshape(-1, V), targets_single[:, :, 0].reshape(-1))

print(f"MTP loss (N=1):  {mtp_loss.item():.6f}")
print(f"Standard CE:     {ce_loss.item():.6f}")
print(f"N=1 matches CE:  {torch.allclose(mtp_loss, ce_loss, atol=1e-5)}")

# N=3 heads
heads_3 = [torch.randn(D, V) for _ in range(3)]
targets_3 = torch.randint(0, V, (B, S, 3))
loss_3 = multi_token_prediction_loss(hidden, heads_3, targets_3)
print(f"MTP loss (N=3):  {loss_3.item():.6f}")

In [ ]:
from torch_judge import check
check("multi_token_prediction")

## 📝 核心思路总结

1. **多头预测**：N 个独立预测头同时预测未来 N 个 token
2. **手写交叉熵**：log-softmax + gather + 取负平均
3. **数值稳定**：先减最大值再 exp，防止溢出
4. **gather 操作**：`log_probs.gather(-1, tgt.unsqueeze(-1))` 取目标 token 的概率